# Cognix Round 2: Analysis

**Runs locally on Mac (no GPU needed).** Uses cached pooled vectors from the overnight TRIBE run.

**What this tests:**
1. Does the Round 1 divergence hold at 1,000 pairs?
2. **Does TRIBE add value beyond raw LLaMA embeddings?** (the critical test)
3. What is the random baseline brain similarity? (quantifies the high-baseline problem)
4. Per-category statistical significance with 25+ pairs each
5. Overfitting checks: text-length correlation, adversarial pairs, permutation tests

In [ ]:
import json
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import cosine as cosine_dist
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from pathlib import Path

In [ ]:
# ---- Config ----
# Adjust these paths based on where you put the cached vectors
PAIRS_PATH = '../data/validation_pairs_r2.jsonl'
POOLED_DIR = Path('../artifacts/r2_pooled_vectors')  # downloaded from Google Drive

def text_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]

## 1. Load pairs and cached brain vectors

In [ ]:
pairs = []
with open(PAIRS_PATH) as f:
    for line in f:
        pairs.append(json.loads(line))

print(f'Total pairs: {len(pairs)}')
sources = {}
for p in pairs:
    src = p['source']
    sources[src] = sources.get(src, 0) + 1
print('Sources:', sources)

categories = {}
for p in pairs:
    cat = p['category']
    categories[cat] = categories.get(cat, 0) + 1
print('Categories:', dict(sorted(categories.items())))

In [ ]:
# Load all cached pooled vectors into a dict
brain_cache = {}
for npy_file in POOLED_DIR.glob('*.npy'):
    h = npy_file.stem
    brain_cache[h] = np.load(npy_file)
print(f'Loaded {len(brain_cache)} cached brain vectors')

## 2. Compute semantic similarity (sentence-transformer)

In [ ]:
st_model = SentenceTransformer('all-MiniLM-L6-v2')

semantic_sims = []
for p in tqdm(pairs, desc='Semantic similarity'):
    emb_a = st_model.encode(p['text_a'])
    emb_b = st_model.encode(p['text_b'])
    sim = 1.0 - cosine_dist(emb_a, emb_b)
    semantic_sims.append(sim)
semantic_sims = np.array(semantic_sims)

## 3. Compute brain similarity (from cached vectors)

In [ ]:
brain_sims = []
valid_mask = []
for p in pairs:
    ha = text_hash(p['text_a'])
    hb = text_hash(p['text_b'])
    va = brain_cache.get(ha)
    vb = brain_cache.get(hb)
    if va is not None and vb is not None:
        sim = 1.0 - cosine_dist(va, vb)
        brain_sims.append(sim)
        valid_mask.append(True)
    else:
        brain_sims.append(np.nan)
        valid_mask.append(False)

brain_sims = np.array(brain_sims)
valid_mask = np.array(valid_mask)
print(f'Brain similarity computed for {valid_mask.sum()}/{len(pairs)} pairs')

## 4. LLaMA baseline (THE CRITICAL TEST)

If raw LLaMA embeddings show the same divergence as brain vectors, TRIBE's brain mapping adds nothing.

We use `all-mpnet-base-v2` as a stronger semantic baseline. For a true LLaMA test, run LLaMA embeddings on Colab and cache them — but this gives us a good proxy.

In [ ]:
# Stronger semantic baseline
st_strong = SentenceTransformer('all-mpnet-base-v2')

strong_sims = []
for p in tqdm(pairs, desc='Strong semantic baseline'):
    emb_a = st_strong.encode(p['text_a'])
    emb_b = st_strong.encode(p['text_b'])
    sim = 1.0 - cosine_dist(emb_a, emb_b)
    strong_sims.append(sim)
strong_sims = np.array(strong_sims)

## 5. Overall correlation — per source

In [ ]:
print('=' * 70)
print('CORRELATION: Brain similarity vs Semantic similarity')
print('=' * 70)

for source_name in ['handcrafted', 'stsb', 'wikipedia_random', 'ALL']:
    if source_name == 'ALL':
        mask = valid_mask
    else:
        mask = np.array([p['source'] == source_name and valid_mask[i] for i, p in enumerate(pairs)])
    
    if mask.sum() < 5:
        continue
    
    sem = semantic_sims[mask]
    brain = brain_sims[mask]
    r_p, p_p = stats.pearsonr(sem, brain)
    r_s, p_s = stats.spearmanr(sem, brain)
    
    print(f'\n  {source_name} (n={mask.sum()})')
    print(f'    Pearson r:  {r_p:.4f}  (p={p_p:.2e})')
    print(f'    Spearman r: {r_s:.4f}  (p={p_s:.2e})')

print('\n' + '=' * 70)
print('CORRELATION: Brain similarity vs Strong semantic baseline (mpnet)')
print('=' * 70)

mask_all = valid_mask
r_p, p_p = stats.pearsonr(strong_sims[mask_all], brain_sims[mask_all])
r_s, p_s = stats.spearmanr(strong_sims[mask_all], brain_sims[mask_all])
print(f'  Pearson r:  {r_p:.4f}  (p={p_p:.2e})')
print(f'  Spearman r: {r_s:.4f}  (p={p_s:.2e})')

## 6. Random baseline distribution (quantifies the high-baseline problem)

In [ ]:
random_mask = np.array([p['source'] == 'wikipedia_random' and valid_mask[i] for i, p in enumerate(pairs)])
random_brain = brain_sims[random_mask]
random_sem = semantic_sims[random_mask]

print('RANDOM WIKIPEDIA PAIRS (baseline brain similarity)')
print(f'  N: {random_mask.sum()}')
print(f'  Brain sim:    mean={np.nanmean(random_brain):.3f}, std={np.nanstd(random_brain):.3f}')
print(f'  Semantic sim: mean={np.nanmean(random_sem):.3f}, std={np.nanstd(random_sem):.3f}')
print()
print(f'  This is the "floor" — the brain similarity you get for arbitrary text pairs.')
print(f'  Any divergence category must score ABOVE this to be meaningful.')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(random_brain[~np.isnan(random_brain)], bins=30, alpha=0.7, label='Brain sim (random pairs)')
ax.axvline(np.nanmean(random_brain), color='red', linestyle='--', label=f'Mean: {np.nanmean(random_brain):.3f}')
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Count')
ax.set_title('Brain similarity distribution for random Wikipedia pairs')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Per-category breakdown (handcrafted pairs only)

In [ ]:
print(f'{"Category":<40} {"N":>3}  {"Sem":>6}  {"Brain":>6}  {"Diff":>6}  {"p-value":>8}')
print('-' * 80)

handcrafted_cats = sorted(set(
    p['category'] for p in pairs if p['source'] == 'handcrafted'
))

for cat in handcrafted_cats:
    mask = np.array([
        p['category'] == cat and p['source'] == 'handcrafted' and valid_mask[i]
        for i, p in enumerate(pairs)
    ])
    if mask.sum() < 3:
        continue
    
    sem = semantic_sims[mask]
    brain = brain_sims[mask]
    diff = brain - sem
    
    # Paired t-test: is brain sim significantly different from semantic sim?
    t_stat, p_val = stats.ttest_rel(brain, sem)
    
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    print(f'{cat:<40} {mask.sum():>3}  {sem.mean():>6.3f}  {brain.mean():>6.3f}  {diff.mean():>+6.3f}  {p_val:>8.2e} {sig}')

## 8. Scatter plot

In [ ]:
category_colors = {
    'paraphrase': '#2ecc71',
    'unrelated': '#95a5a6',
    'cognitive_load': '#e74c3c',
    'emotional_arousal': '#e67e22',
    'sensorimotor': '#9b59b6',
    'spatial_scene': '#3498db',
    'syntactic_surprise': '#1abc9c',
    'narrative_suspense': '#f39c12',
    'theory_of_mind': '#e91e63',
    'control_same_topic_diff_processing': '#607d8b',
    'adversarial': '#000000',
    'stsb': '#bdc3c7',
    'random_baseline': '#dfe6e9',
}

fig, ax = plt.subplots(figsize=(12, 9))

for cat in category_colors:
    mask = np.array([p['category'] == cat and valid_mask[i] for i, p in enumerate(pairs)])
    if mask.sum() == 0:
        continue
    ax.scatter(
        semantic_sims[mask], brain_sims[mask],
        c=category_colors[cat], label=cat, alpha=0.6, s=30,
        edgecolors='white', linewidth=0.3,
    )

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='y=x')

mask_all = valid_mask
r_p, _ = stats.pearsonr(semantic_sims[mask_all], brain_sims[mask_all])
r_s, _ = stats.spearmanr(semantic_sims[mask_all], brain_sims[mask_all])

ax.set_xlabel('Semantic Similarity', fontsize=12)
ax.set_ylabel('Brain Similarity (TRIBE v2)', fontsize=12)
ax.set_title(f'Cognix Round 2: Semantic vs Brain Similarity\nPearson r={r_p:.3f}, Spearman r={r_s:.3f} (n={mask_all.sum()})', fontsize=14)
ax.legend(loc='upper left', fontsize=7, ncol=2)
ax.set_xlim(-0.15, 1.05)
ax.set_ylim(-0.15, 1.05)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('r2_scatter.png', dpi=150)
plt.show()

## 9. Overfitting checks

In [ ]:
# ---- Check 1: Text length correlation ----
# Is brain similarity just a proxy for text length?

len_a = np.array([len(p['text_a'].split()) for p in pairs])
len_b = np.array([len(p['text_b'].split()) for p in pairs])
len_diff = np.abs(len_a - len_b)
len_mean = (len_a + len_b) / 2

mask_v = valid_mask
r_len_brain, p_len_brain = stats.pearsonr(len_mean[mask_v], brain_sims[mask_v])
r_lendiff_brain, p_lendiff_brain = stats.pearsonr(len_diff[mask_v], brain_sims[mask_v])

print('TEXT LENGTH vs BRAIN SIMILARITY')
print(f'  Mean text length vs brain sim:  r={r_len_brain:.3f}  (p={p_len_brain:.2e})')
print(f'  Length difference vs brain sim: r={r_lendiff_brain:.3f}  (p={p_lendiff_brain:.2e})')
print()
if abs(r_len_brain) > 0.5:
    print('  WARNING: Strong correlation between text length and brain similarity.')
    print('  The brain signal may be confounded with text length.')
else:
    print('  Text length is not strongly correlated with brain similarity. Good.')

In [ ]:
# ---- Check 2: Adversarial pairs ----
adv_mask = np.array([p['category'] == 'adversarial' and valid_mask[i] for i, p in enumerate(pairs)])

if adv_mask.sum() > 0:
    print('ADVERSARIAL PAIRS')
    print(f'  N: {adv_mask.sum()}')
    for i, p in enumerate(pairs):
        if p['category'] == 'adversarial' and valid_mask[i]:
            print(f'  id={p["id"]}  expected={p["expected"]}  sem={semantic_sims[i]:.3f}  brain={brain_sims[i]:.3f}')
            print(f'    A: {p["text_a"][:80]}...')
            print(f'    B: {p["text_b"][:80]}...')
            print()

In [ ]:
# ---- Check 3: Permutation test ----
# Shuffle category labels and recompute per-category divergence.
# If the real divergence is within the shuffled distribution, it's not meaningful.

handcrafted_mask = np.array([p['source'] == 'handcrafted' and valid_mask[i] for i, p in enumerate(pairs)])
hc_brain = brain_sims[handcrafted_mask]
hc_sem = semantic_sims[handcrafted_mask]
hc_cats = [p['category'] for i, p in enumerate(pairs) if p['source'] == 'handcrafted' and valid_mask[i]]

# Real mean divergence for divergence categories
divergence_cats = {'cognitive_load', 'emotional_arousal', 'sensorimotor', 'spatial_scene',
                   'syntactic_surprise', 'narrative_suspense', 'theory_of_mind'}
div_mask = np.array([c in divergence_cats for c in hc_cats])
real_div_mean = (hc_brain[div_mask] - hc_sem[div_mask]).mean()

# Permutation: shuffle categories 1000 times
rng = np.random.default_rng(42)
n_perms = 1000
perm_divs = []
for _ in range(n_perms):
    shuffled = rng.permutation(hc_cats)
    perm_div_mask = np.array([c in divergence_cats for c in shuffled])
    perm_div = (hc_brain[perm_div_mask] - hc_sem[perm_div_mask]).mean()
    perm_divs.append(perm_div)

perm_divs = np.array(perm_divs)
p_perm = (perm_divs >= real_div_mean).mean()

print('PERMUTATION TEST')
print(f'  Real mean divergence (divergence categories): {real_div_mean:.3f}')
print(f'  Permuted mean divergence: mean={perm_divs.mean():.3f}, std={perm_divs.std():.3f}')
print(f'  Permutation p-value: {p_perm:.4f}')
if p_perm < 0.05:
    print('  The divergence is significant beyond what shuffled labels would produce.')
else:
    print('  WARNING: The divergence is not significant under permutation testing.')

## 10. Top 30 most divergent pairs

In [ ]:
divergence = brain_sims - semantic_sims
valid_indices = np.where(valid_mask)[0]
sorted_idx = valid_indices[np.argsort(divergence[valid_indices])[::-1]]

print('TOP 30: Brain says similar, semantics says not')
print('=' * 90)
for rank, idx in enumerate(sorted_idx[:30], 1):
    p = pairs[idx]
    print(f'\n#{rank} (id={p["id"]}, src={p["source"]}, cat={p["category"]})')
    print(f'  Sem: {semantic_sims[idx]:.3f}  Brain: {brain_sims[idx]:.3f}  Div: {divergence[idx]:+.3f}')
    print(f'  A: {p["text_a"][:100]}...')
    print(f'  B: {p["text_b"][:100]}...')

## 11. Summary

In [ ]:
mask_all = valid_mask
r_p, p_p = stats.pearsonr(semantic_sims[mask_all], brain_sims[mask_all])
r_s, p_s = stats.spearmanr(semantic_sims[mask_all], brain_sims[mask_all])

print('ROUND 2 SUMMARY')
print('=' * 60)
print(f'Total pairs: {len(pairs)}')
print(f'Valid pairs: {valid_mask.sum()}')
print(f'Pearson r (overall):  {r_p:.4f}  (p={p_p:.2e})')
print(f'Spearman r (overall): {r_s:.4f}  (p={p_s:.2e})')
print()
print(f'Random baseline brain sim: {np.nanmean(random_brain):.3f} +/- {np.nanstd(random_brain):.3f}')
print(f'Text-length correlation: r={r_len_brain:.3f}')
print(f'Permutation test p-value: {p_perm:.4f}')
print()
print('NEXT STEPS:')
print('- If r < 0.4 and permutation p < 0.05: signal is real at scale')
print('- Check LLaMA baseline: if brain divergence >> semantic divergence, TRIBE adds value')
print('- Check random baseline: this is the floor for baseline removal in Phase 3')
print('- Proceed to Phase 3 (baseline removal) and Phase 4 (region validity test)')

In [ ]:
# ---- Save results ----
results = []
for i, p in enumerate(pairs):
    results.append({
        'id': p['id'],
        'source': p['source'],
        'category': p['category'],
        'expected': p['expected'],
        'sim_semantic': float(semantic_sims[i]),
        'sim_brain': float(brain_sims[i]) if valid_mask[i] else None,
        'sim_strong_semantic': float(strong_sims[i]),
        'text_a_len': len(p['text_a'].split()),
        'text_b_len': len(p['text_b'].split()),
    })

with open('../data/r2_results.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print('Results saved to data/r2_results.jsonl')